In [1]:
import sys
project_root = "/home/krosen5/Court Case Sequence Transformer Model/court_case_transformer_project"

if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(sys.path[:3])

['/home/krosen5/Court Case Sequence Transformer Model/court_case_transformer_project', '/home/krosen5/anaconda3/envs/legalsequencetransformer/lib/python313.zip', '/home/krosen5/anaconda3/envs/legalsequencetransformer/lib/python3.13']


In [2]:
import os
print(os.getcwd())
print(os.listdir("."))

/home/krosen5/Court Case Sequence Transformer Model/court_case_transformer_project/side_tests
['load_in_model_just_clustering.ipynb', 'load_in_model_just_confident_misclassifications.ipynb', '__init__.py', 'work3.ipynb', '.ipynb_checkpoints', 'load_in_model_complete_some_clustering.ipynb', 'load_in_model_just_clustering_DV_1.ipynb', 'work2.ipynb', 'work.ipynb', 'work4.ipynb', '__pycache__', 'load_in_model_complete.ipynb']


In [3]:
import pickle
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans, DBSCAN

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import math

In [4]:
from utils.cache_directory_utils import CacheDirectoryManager
from data.load_data import mount_drive, load_data
from models.transformer_model import CaseTransformerModel
from models.transformer_creation import create_vocabularies_from_data
from config import get_model_config
from analysis.pca_vectors import (
    extract_all_pre_classifier_vectors,
    plot_multilabel_cluster_heatmap,
)
from notebook_helpers import prepare_sequences_and_labels

In [5]:
BASE_CACHE  = Path('/data/krosen5/cache/transformers/balanced_8')
MODELS_DIR  = BASE_CACHE / 'saved_models'
OUTPUT_DIR  = BASE_CACHE / 'pca_best_fold'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BEST_FOLD_IDX = 0        # 0-indexed - change to your best fold
N_FOLDS       = 5
RANDOM_SEED   = 42
N_CLUSTERS    = 6
bert_out_dim  = 768
cuda_index    = 0

In [6]:
computation_device = torch.device(
    f'cuda:{cuda_index}' if torch.cuda.is_available() else 'cpu'
)
print(f'Using device: {computation_device}')

model_cache_manager = CacheDirectoryManager(base_path=BASE_CACHE)
model_config = get_model_config()

with open(MODELS_DIR / 'label_processor.pkl', 'rb') as f:
    label_processor = pickle.load(f)
print(f'Loaded label processor ({len(label_processor.label_names)} labels)')

sequences_path = model_cache_manager.get_cache_dir('datasets') / 'sequences_and_labels.pkl'
with open(sequences_path, 'rb') as f:
    sequences, label_sequences_raw = pickle.load(f)
print(f'Loaded {len(sequences)} sequences')

filtered_training_data = list(zip(sequences, label_sequences_raw))
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)
fold_val_sets = [
    [filtered_training_data[i] for i in val_idx]
    for _, val_idx in kf.split(filtered_training_data)
]
print(f'Recreated {N_FOLDS} fold splits')
print(f'   Best fold {BEST_FOLD_IDX + 1} val size: {len(fold_val_sets[BEST_FOLD_IDX])}')

Using device: cuda:0
Loaded label processor (20 labels)
Loaded 2533 sequences
Recreated 5 fold splits
   Best fold 1 val size: 507


In [7]:
parameters_dict = {
    'model_validation_strategy': 'k_fold_cross_validation',
    'k_fold_num_folds': 5,
    'k_fold_shuffle': True,
    'random_seed': 42,
    'skip_duplicate_labels': False,
    'skip_duplicate_labels_on_import': False,
    'bert_model_to_use': 'bert-base-uncased',
    'bert_batch_size': 32,
    'sampling_strategy': 'balanced',
    'weighted_loss_function': False,
    'use_mixed_precision': True,
    'use_learning_rate_scheduler': True,
    'gradient_clip_norm': 1.0,
    'number_of_epochs_for_training': 20,
    'batch_size_for_training': 16,
    'low_frequency_threshold': 400,
    'folder_name_directory': 'balanced_8',
}

In [8]:
drive_root_folder = mount_drive('tulane-courts-shared/v2')
court_data = load_data(drive_root_folder)
charge_vocab, mag_vocab = create_vocabularies_from_data(court_data)
del court_data


[OK] Found Google Drive mount at /data/krosen5/gdrive-all
[OK] Google Drive mount is accessible
[OK] Data directory found: /data/krosen5/gdrive-all/tulane-courts-shared/v2
Creating charge vocabulary from charges DataFrame with 2139 unique charges
Found 5 unique magistrate sections in mag data.
Creating magistrate vocabulary of size 5


In [9]:
bond_dim = model_config.NUM_BOND_CLASSES  # 9
sample_pt = MODELS_DIR / f'k_fold_fold_{BEST_FOLD_IDX + 1}' / \
            f'final_model_mlb_fold_{BEST_FOLD_IDX + 1}.pt'
sample_sd = torch.load(sample_pt, map_location='cpu')
fuse_in   = sample_sd['fuse.0.weight'].shape[1]  # 1548
FIXED_DIMS = 32 + 192 + 768 + 32 + 96 + 192 + 1 + 1  # 1314
correct_temporal_dim = fuse_in - FIXED_DIMS - (bond_dim * 2)  # 216
print(f'temporal_dim={correct_temporal_dim}')

temporal_dim=216


In [10]:
print(f'\n{"-"*60}')
print(f'  Loading fold {BEST_FOLD_IDX + 1} (best fold)')
print(f'{"-"*60}')

model = CaseTransformerModel(
    charge_vocab=charge_vocab,
    mag_vocab=mag_vocab,
    bert_cache_dir=model_cache_manager.get_cache_dir('bert'),
    bert_out_dim=bert_out_dim,
    num_labels=model_config.num_labels,
    bond_dim=bond_dim,
    temporal_dim=correct_temporal_dim,
    cache_manager=model_cache_manager,
)
model.load_state_dict(torch.load(sample_pt, map_location=computation_device))
model.eval()
model.to(computation_device)
print(f'Loaded {sample_pt.name}')


------------------------------------------------------------
  Loading fold 1 (best fold)
------------------------------------------------------------


Loaded final_model_mlb_fold_1.pt


In [11]:
_saved_min_year = model_config.min_year
_saved_max_year = model_config.max_year
model_config.update_date_range(2008, 2025)
print(f'  Config forced: temporal_dim={model_config.temporal_dim}')

try:
    vectors, final_labels, case_labels_matrix, metadata = \
        extract_all_pre_classifier_vectors(
            model,
            fold_val_sets[BEST_FOLD_IDX],
            label_processor,
            computation_device,
            use_teacher_forcing=True,
            verbose=True,
        )
finally:
    model_config.update_date_range(_saved_min_year, _saved_max_year)
    print(f'Config restored: temporal_dim={model_config.temporal_dim}')

del model
torch.cuda.empty_cache()

print(f'\nExtracted {vectors.shape[0]} embeddings  (dim={vectors.shape[1]})')

Date range updated: 2008 to 2025
  Config forced: temporal_dim=216


Extracting pre-classifier vectors for 507 samples: 100%|##########| 507/507 [00:04<00:00, 105.41it/s]

Extracted 507 vectors of dimension 896
Date range updated: 1990 to 2025
Config restored: temporal_dim=432

Extracted 507 embeddings  (dim=896)


In [12]:
SHORT_LABEL_NAMES = {
    "FIRST_APPEARANCE_HEARING_OCCURRED": "First Appearance Hearing",
    "BOND_SET": "Bond Set",
    "BOND_NOT_SET": "Bond Not Set",
    "DEFENDANT_RELEASED_ON_OWN_RECOGNIZANCE": "Released, ROR",
    "THE_COURT_APPOINTED_THE_ORLEANS_PUBLIC_DEFENDER": "Public Defender Appointed",
    "ANKLE_MONITORING_IMPOSED": "Ankle Monitor",
    "DOMESTIC_STAY_AWAY_ORDER_IMPOSED": "Domestic Stay-Away Order",
    "CHARGES_ACCEPTED": "Charges Accepted",
    "CHARGES_REFUSED": "Charges Refused",
    "PROBABLE_CAUSE_FOUND": "PC Found",
    "PROBABLE_CAUSE_NOT_FOUND": "PC Not Found",
    "DEFENDANT_IN_CUSTODY_AND_NOT_IN_COURT": "In Custody, Absent",
    "THE_STATE_WAS_NOT_PRESENT": "State Not Present",
    "PRE_TRIAL_CONFERENCE_HELD": "Pre-Trial Conference",
    "CONTINUANCE_OCCURRED": "Continuance",
    "PLED_GUILTY_TO_CHARGES": "Pled Guilty",
    "PLED_NOT_GUILTY_TO_CHARGES": "Pled Not Guilty",
    "TRIAL_VERDICT_REACHED": "Trial Verdict",
    "SENTENCE_THAT_INCLUDES_JAIL_TIME_WAS_IMPOSED": "Sentence - Jail",
    "SENTENCE_THAT_DOES_NOT_INCLUDE_JAIL_TIME_WAS_IMPOSED": "Sentence - No Jail",
    "FOUND_GUILTY_BY_JUDGE_OR_JURY": "Found Guilty",
    "FOUND_NOT_GUILTY_BY_JUDGE_OR_JURY": "Found Not Guilty",
}

In [13]:
vectors = StandardScaler().fit_transform(vectors)
vectors = normalize(vectors, norm='l2')
print('Normalised (StandardScaler + L2)')

case_labels_arr = np.asarray(case_labels_matrix)

Normalised (StandardScaler + L2)


In [14]:
print('\nRunning PCA (50 components)...')
pca = PCA(n_components=50, random_state=RANDOM_SEED)
vecs_50d = pca.fit_transform(vectors)
explained = pca.explained_variance_ratio_.sum() * 100
print(f'PCA done - {explained:.1f}% variance in 50 PCs')

print('Running t-SNE...')
projected_2d = TSNE(
    n_components=2,
    perplexity=40,
    learning_rate='auto',
    init='pca',
    random_state=RANDOM_SEED,).fit_transform(vecs_50d)
print(f't-SNE done - shape: {projected_2d.shape}')



Running PCA (50 components)...
PCA done - 93.3% variance in 50 PCs
Running t-SNE...
t-SNE done - shape: (507, 2)


In [15]:
km = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_SEED, n_init=10)
km_labels = km.fit_predict(projected_2d)
print(f'KMeans: {N_CLUSTERS} clusters')

KMeans: 6 clusters


In [16]:
def scatter_2d(xy, colours, legend_labels, title, output_path,
               xlabel='t-SNE dim 1', ylabel='t-SNE dim 2'):

    # -- appendix single-column: full text width = 7.16" ---------------------
    plt.rcParams.update({
        'font.family':        'serif',
        'font.serif':         ['Times New Roman', 'Times', 'DejaVu Serif'],
        'font.size':          11,
        'axes.titlesize':     12,
        'axes.labelsize':     11,
        'xtick.labelsize':    10,
        'ytick.labelsize':    10,
        'legend.fontsize':    10,
        'axes.linewidth':     0.8,
        'xtick.major.width':  0.7,
        'ytick.major.width':  0.7,
        'xtick.major.size':   4,
        'ytick.major.size':   4,
        'xtick.minor.visible': False,
        'ytick.minor.visible': False,
        'pdf.fonttype':       42,
        'ps.fonttype':        42,
    })

    # full IEEE text width; golden-ratio height (~4.4") is clean for scatter
    width  = 7.16
    height = width / 1.618   # ~ 4.43"

    fig, ax = plt.subplots(figsize=(width, height), constrained_layout=True)

    unique = sorted(set(colours))
    cmap   = cm.get_cmap('tab10', len(unique))

    for i, val in enumerate(unique):
        mask  = np.array(colours) == val
        label = legend_labels[i] if legend_labels else str(val)
        ax.scatter(
            xy[mask, 0], xy[mask, 1],
            c=[cmap(i)], label=label,
            s=25, alpha=0.85,
            edgecolors='white', linewidths=0.3,
            rasterized=True,
        )

    ax.set_xlabel(xlabel, fontsize=11, labelpad=5)
    ax.set_ylabel(ylabel, fontsize=11, labelpad=5)
    ax.set_title(title,   fontsize=12, fontweight='bold', pad=6)

    ax.tick_params(axis='both', which='major',
                   labelsize=10, length=4, width=0.7, pad=4)

    for spine in ax.spines.values():
        spine.set_linewidth(0.8)

    ax.legend(
        loc='best',
        fontsize=10,
        frameon=True,
        edgecolor='grey',
        fancybox=False,
        markerscale=1.3,
        handletextpad=0.5,
        borderpad=0.6,
        labelspacing=0.4,
        ncols = 2,
    )

    fig.savefig(output_path, dpi=600, bbox_inches='tight', format='png')
    print(f'Saved: {output_path}')
    plt.close(fig)

def plot_label_grid_colored_by_cluster(
    projected_2d, label_matrix, label_names, cluster_labels, output_path,
    title='t-SNE Embeddings (Best Fold) - Label Presence', max_cols=5,
):
    label_matrix   = np.asarray(label_matrix)
    cluster_labels = np.asarray(cluster_labels)
    n_samples, n_labels = label_matrix.shape
    unique_clusters = sorted(np.unique(cluster_labels))
    n_real = sum(1 for c in unique_clusters if c != -1)
    cmap = cm.get_cmap('tab20' if n_real <= 20 else 'nipy_spectral', max(n_real, 1))
    cl_to_idx = {cl: i for i, cl in enumerate(unique_clusters)}

    n_cols = min(max_cols, n_labels)
    n_rows = math.ceil(n_labels / n_cols)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
    axes = np.atleast_1d(axes).flatten()

    for i, name in enumerate(label_names):
        ax = axes[i]
        mask_pos = label_matrix[:, i] == 1
        n_pos = int(mask_pos.sum())
        pct = 100 * n_pos / n_samples if n_samples else 0.0
        ax.scatter(projected_2d[~mask_pos, 0], projected_2d[~mask_pos, 1],
                   c='lightgrey', s=8, alpha=0.25, edgecolors='none')
        for cl in unique_clusters:
            mask = mask_pos & (cluster_labels == cl)
            if not mask.any():
                continue
            fc = 'dimgrey' if cl == -1 else [cmap(cl_to_idx[cl])]
            ax.scatter(projected_2d[mask, 0], projected_2d[mask, 1],
                       c=fc, s=14, alpha=0.35 if cl == -1 else 0.75,
                       edgecolors='white', linewidths=0.2)
        ax.set_title(f'{name[:30]}\n(n={n_pos:,}, {pct:.1f}%)', fontsize=8)
        ax.set_xticks([]); ax.set_yticks([])

    for j in range(n_labels, len(axes)):
        axes[j].set_visible(False)

    handles = [
        plt.Line2D([0], [0], marker='o', color='w',
                   markerfacecolor='dimgrey' if cl == -1 else cmap(cl_to_idx[cl]),
                   markeredgecolor='white', markersize=7,
                   label='Noise' if cl == -1 else f'Cluster {cl}')
        for cl in unique_clusters
    ]
    fig.legend(handles=handles, loc='upper center',
               bbox_to_anchor=(0.5, 1.01),
               ncol=min(len(unique_clusters), 6), fontsize=9)
    fig.suptitle(title, fontsize=14, fontweight='bold', y=1.05)
    fig.tight_layout()
    fig.savefig(output_path, dpi=200, bbox_inches='tight')
    print(f'Saved: {output_path}')
    plt.close(fig)

In [17]:
fold_label = f'Fold {BEST_FOLD_IDX + 1} (best)'

scatter_2d(
    projected_2d,
    km_labels.tolist(),
    [f'Cluster {i}' for i in range(N_CLUSTERS)],
    title=f't-SNE Embeddings - KMeans Clusters ({fold_label})',
    output_path=OUTPUT_DIR / 'tsne_kmeans_clusters.png',
)

plot_label_grid_colored_by_cluster(
    projected_2d=projected_2d,
    label_matrix=case_labels_arr,
    label_names=label_processor.label_names,
    cluster_labels=km_labels,
    output_path=OUTPUT_DIR / 'tsne_label_grid.png',
    title=f't-SNE Embeddings ({fold_label}) - Label Presence by Cluster',
)

plot_multilabel_cluster_heatmap(
    km_labels, case_labels_arr, label_processor.label_names,
    output_path=OUTPUT_DIR / 'tsne_multilabel_heatmap.png',
    label_name_map=SHORT_LABEL_NAMES,
)

print(f'\nDone. All plots saved to {OUTPUT_DIR}')
print(f'   Best fold: {BEST_FOLD_IDX + 1}')
print(f'   Samples:   {projected_2d.shape[0]}')
print(f'   PCA var:   {explained:.1f}%')



Saved: /data/krosen5/cache/transformers/balanced_8/pca_best_fold/tsne_kmeans_clusters.png
Saved: /data/krosen5/cache/transformers/balanced_8/pca_best_fold/tsne_label_grid.png
Saved multi-label heatmap to /data/krosen5/cache/transformers/balanced_8/pca_best_fold/tsne_multilabel_heatmap.png

MULTI-LABEL CLUSTER SUMMARY
----------------------------------------------------------------------
  Cluster 0 (75 samples): FIRST_APPEARANCE_HEARING_OCCURRED (100%, n=75), BOND_SET (95%, n=71), CHARGES_REFUSED (77%, n=58), THE_COURT_APPOINTED_THE_ORLEANS_PUBLIC_DEFENDER (68%, n=51), CONTINUANCE_OCCURRED (39%, n=29), DOMESTIC_STAY_AWAY_ORDER_IMPOSED (28%, n=21), PROBABLE_CAUSE_FOUND (28%, n=21), DEFENDANT_RELEASED_ON_OWN_RECOGNIZANCE (25%, n=19), PROBABLE_CAUSE_NOT_FOUND (21%, n=16), CHARGES_ACCEPTED (19%, n=14), PLED_GUILTY_TO_CHARGES (19%, n=14), THE_STATE_WAS_NOT_PRESENT (16%, n=12), BOND_NOT_SET (16%, n=12), SENTENCE_THAT_INCLUDES_JAIL_TIME_WAS_IMPOSED (12%, n=9), ANKLE_MONITORING_IMPOSED (9%, n=